# TryOn Studio — fresh Colab GPU validation

Run cells in order. Select **Runtime → Change runtime type → T4 GPU** first. Add a `GITHUB_TOKEN` secret in Colab with **Contents: Read-only** access to the private project repository, and enable notebook access to it.

This notebook does not start a public API/tunnel. It prepares Python 3.11 separately from Colab's kernel, then previews a simple clothing mask and runs one real CatVTON generation. The rectangle mask is only a technical acceptance test, not the final automatic masker. Photos and results stay on the VM, not Drive. Use images you have permission to process.


## 1. GPU preflight


In [ ]:
import subprocess, sys
gpu = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
                     capture_output=True, text=True)
if gpu.returncode:
    raise RuntimeError('No NVIDIA GPU. Select Runtime > Change runtime type > T4 GPU.')
print('Kernel Python:', sys.version.split()[0])
print('GPU:', gpu.stdout.strip())


## 2. Mount cache Drive and fetch source

Drive stores only reusable package/model caches and the setup log. Source code lives on the VM's faster local disk. You do not need to make a Drive folder. On a new runtime, rerun cells 1–3.


In [ ]:
from google.colab import drive, userdata
from getpass import getpass
from pathlib import Path
import base64, os, subprocess, tempfile

RUNTIME = Path('/content/tryon-studio')
SOURCE = RUNTIME / 'source'
PERSISTENT = Path('/content/drive/MyDrive/TryOnStudio')
RUNTIME.mkdir(parents=True, exist_ok=True)
drive.mount('/content/drive')
PERSISTENT.mkdir(parents=True, exist_ok=True)

def sync_source():
    try:
        secret = userdata.get('GITHUB_TOKEN')
    except Exception:
        secret = None
    secret = (secret or getpass('GITHUB_TOKEN (hidden): ')).strip()
    if not secret:
        raise RuntimeError('GITHUB_TOKEN is empty; grant Contents: Read-only access to the private repository.')
    encoded = base64.b64encode(('x-access-token:' + secret).encode()).decode()
    env = {**os.environ, 'GIT_CONFIG_COUNT': '1',
           'GIT_CONFIG_KEY_0': 'http.https://github.com/.extraHeader',
           'GIT_CONFIG_VALUE_0': 'Authorization: Basic ' + encoded,
           'GIT_TERMINAL_PROMPT': '0'}
    def git(*args):
        try:
            result = subprocess.run(['git', *map(str, args)], env=env,
                                    capture_output=True, text=True, timeout=300)
        except subprocess.TimeoutExpired:
            raise RuntimeError('Git timed out after 5 minutes. Retry this cell.') from None
        if result.returncode:
            error = (result.stderr or result.stdout).replace(secret, '[REDACTED]').replace(encoded, '[REDACTED]')
            raise RuntimeError('Git failed: ' + error.strip())
        return result.stdout.strip()
    if SOURCE.exists() and not (SOURCE / '.git').is_dir():
        raise RuntimeError('Local source path exists without Git metadata; inspect it. No files were deleted.')
    if not SOURCE.exists():
        print('Cloning source onto local VM...', flush=True)
        with tempfile.TemporaryDirectory(dir=RUNTIME) as staging:
            checkout = Path(staging) / 'checkout'
            git('clone', '--single-branch',
                'https://github.com/CloudyPatil/ai-virtual-try-on-extension.git', checkout)
            checkout.rename(SOURCE)
    else:
        print('Fetching project fixes...', flush=True)
        git('-C', SOURCE, 'fetch', 'origin', 'main')
        git('-C', SOURCE, 'merge', '--ff-only', 'FETCH_HEAD')
    print('Source commit:', git('-C', SOURCE, 'rev-parse', '--short', 'HEAD'))
sync_source()


## 3. Prepare and validate environment

First CUDA wheel download is large. Output streams live. This cell is safe to rerun after interruption. **[4/4] Environment preparation complete** means imports passed; it does not mean a model result exists yet.


In [ ]:
from collections import deque
from pathlib import Path
import os, subprocess, sys

RUNTIME = Path('/content/tryon-studio')
SOURCE = RUNTIME / 'source'
PERSISTENT = Path('/content/drive/MyDrive/TryOnStudio')
bootstrap = SOURCE / 'services/inference-worker/colab_bootstrap.py'
if not bootstrap.is_file():
    raise RuntimeError('Project source missing; run cell 2 first.')
env = {**os.environ, 'HF_HOME': str(PERSISTENT / 'huggingface'),
       'HF_HUB_CACHE': str(PERSISTENT / 'huggingface/hub'), 'PYTHONUNBUFFERED': '1'}
command = [sys.executable, '-u', str(bootstrap), '--project-root', str(SOURCE),
           '--persistent-root', str(PERSISTENT), '--prepare-only']
tail = deque(maxlen=30)
with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
        tail.append(line)
    code = process.wait()
if code:
    raise RuntimeError(f'Setup failed (exit {code}); run diagnostics. Last output:\n' + ''.join(tail))


## 4. Upload two images and preview the clothing mask

Upload one person photo, then one garment photo. Use clear images at least 256 px on each side. Set `CATEGORY` to `upper`, `lower`, or `overall`. If the purple overlay misses the intended clothing, edit `MASK_BOX` as `left,top,right,bottom` fractions from 0 to 1, then rerun this cell. This mask is intentionally rough.


In [ ]:
from google.colab import files
from pathlib import Path
from IPython.display import display, Image as DisplayImage
import subprocess

CATEGORY = 'upper'
MASK_BOX = None  # Example: '0.16,0.21,0.84,0.66'
inputs = Path('/content/tryon-studio/user-input')
inputs.mkdir(parents=True, exist_ok=True)
def upload_one(label, destination):
    print('Upload one ' + label + ' image:')
    items = files.upload()
    if len(items) != 1:
        raise RuntimeError('Select exactly one image for ' + label)
    data = next(iter(items.values()))
    if len(data) > 10 * 1024 * 1024:
        raise RuntimeError(label + ' image exceeds 10 MB')
    destination.write_bytes(data)
person = inputs / 'person.upload'
garment = inputs / 'garment.upload'
upload_one('person', person)
upload_one('garment', garment)
preview = inputs / 'mask-preview.png'
python = Path('/content/tryon-studio/venv-py311/bin/python')
source = Path('/content/tryon-studio/source')
script = source / 'services/inference-worker/colab_tryon.py'
repo = Path('/content/tryon-studio/vendor/CatVTON')
if not python.is_file() or not script.is_file() or not repo.is_dir():
    raise RuntimeError('Environment is missing; run cells 2 and 3 first')
tryon_command = [str(python), '-u', str(script), '--catvton-repo', str(repo),
                 '--person', str(person), '--garment', str(garment),
                 '--category', CATEGORY, '--preview', str(preview)]
if MASK_BOX is not None:
    tryon_command += ['--box', MASK_BOX]
result = subprocess.run(tryon_command, capture_output=True, text=True)
print(result.stdout, end='')
if result.returncode:
    raise RuntimeError('Mask preview failed:\n' + result.stderr[-6000:])
display(DisplayImage(filename=str(preview)))


## 5. Generate one real try-on

Proceed only after inspecting the mask. First run downloads several GB of model weights to the Drive-backed Hugging Face cache. New VMs reload them into GPU memory, but can reuse cached downloads. If interrupted, rerun cells 1–3, then 4–5.


In [ ]:
from IPython.display import display, Image as DisplayImage
from pathlib import Path
import os, subprocess

output = Path('/content/tryon-studio/user-input/tryon-result.png')
if not Path('/content/tryon-studio/user-input/mask-preview.png').is_file():
    raise RuntimeError('Preview the mask in cell 4 first')
env = {**os.environ,
       'HF_HOME': '/content/drive/MyDrive/TryOnStudio/huggingface',
       'HF_HUB_CACHE': '/content/drive/MyDrive/TryOnStudio/huggingface/hub',
       'PYTHONUNBUFFERED': '1'}
command = [*tryon_command, '--output', str(output), '--steps', '25']
with subprocess.Popen(command, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                      text=True, encoding='utf-8', errors='replace', bufsize=1) as process:
    for line in process.stdout:
        print(line, end='', flush=True)
    code = process.wait()
if code:
    raise RuntimeError(f'Generation failed (exit {code}); run diagnostics and send the final error lines')
display(DisplayImage(filename=str(output)))
print('Real CatVTON result generated on GPU; output stayed in this VM.')


## Diagnostics — run after any failure


In [ ]:
from pathlib import Path
import subprocess
runtime = Path('/content/tryon-studio')
log = runtime / 'setup.log'
if not log.is_file():
    log = Path('/content/drive/MyDrive/TryOnStudio/setup.log')
print('Setup log:', log)
print(log.read_text(encoding='utf-8', errors='replace')[-12000:] if log.is_file() else 'Not created yet')
print('VM setup markers:', [p.name for p in runtime.glob('*.ready')])
worker_python = runtime / 'venv-py311/bin/python'
if worker_python.is_file():
    result = subprocess.run([str(worker_python), '-m', 'pip', 'check'], capture_output=True, text=True)
    print('pip check:', result.stdout.strip() or result.stderr.strip())
else:
    print('Worker Python not installed yet')
print('Before sharing output, remove tokens and personal image paths.')
